# 🔴 Interview: FSDP Training Step

Simulated FSDP: all-gather → forward/backward → reduce-scatter → update

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def fsdp_step(param_shards, grad_fn, world_size):
    """
    手写 FSDP 训练步骤 —— 面试版。

    面试考点:
    1. FSDP = Fully Sharded Data Parallel，参数完全分片
    2. All-Gather: 拼接分片得到完整参数 (前向前)
    3. Reduce-Scatter: 梯度分片 (反向后)
    4. 与 DDP 的区别: DDP 每个 worker 有完整参数，FSDP 只有分片

    参数: param_shards: list of 1D tensors, grad_fn: callable, world_size: int
    返回: list of updated 1D tensors
    """
    # Step 1: All-Gather — 拼接所有分片得到完整参数
    full_param = torch.cat(param_shards)  # (total_size,)

    # Step 2: 前向 + 反向 — 用完整参数计算梯度
    grad = grad_fn(full_param)  # (total_size,)

    # Step 3: Reduce-Scatter — 梯度分片
    grad_shards = list(grad.chunk(world_size))  # world_size 个 (shard_size,)

    # Step 4: SGD 更新 — 每个 worker 更新自己的分片
    new_shards = [
        param_shards[i] - 0.01 * grad_shards[i]
        for i in range(world_size)
    ]
    return new_shards

In [ ]:
# Verify: one FSDP step should match equivalent SGD on the full parameter
torch.manual_seed(0)
world_size = 4
shard_size = 8

# Initial shards
param_shards = [torch.randn(shard_size) for _ in range(world_size)]

# Simple quadratic loss gradient: grad = 2 * param
grad_fn = lambda p: 2.0 * p

# FSDP step
new_shards = fsdp_step(param_shards, grad_fn, world_size)
fsdp_result = torch.cat(new_shards)

# Reference: plain SGD on full param
full_param = torch.cat(param_shards)
ref_result = full_param - 0.01 * grad_fn(full_param)

print("FSDP result shape:", fsdp_result.shape)  # expect (32,)
print("Max diff vs SGD reference:", (fsdp_result - ref_result).abs().max().item())  # expect ~0

In [ ]:
from torch_judge import check
check("fsdp_step")

## 📝 核心思路总结

1. **FSDP 的核心**：参数完全分片存储，前向前 all-gather 拼接，反向后 reduce-scatter 分发梯度。
2. **All-Gather + Reduce-Scatter**：这两个集合通信操作是 FSDP 的灵魂，分别对应前向和反向的参数/梯度需求。
3. **与 DDP 的区别**：DDP 每个 worker 存完整参数（只分片梯度），FSDP 连参数都分片，内存效率更高。
4. **分片更新**：每个 worker 只更新自己持有的参数分片，更新后的分片列表就是下一轮的输入。